<a href="https://colab.research.google.com/github/Adheera13/MusAIc/blob/main/MusAIc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# install the small HF helper (if not already available)
!pip install -q huggingface_hub

from huggingface_hub import notebook_login
notebook_login()   # a widget will pop up; paste your HF access token

In [ ]:
!pip install -q torch torchaudio
!pip install -q diffusers==0.31.0 transformers accelerate
!pip install torchsde

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 5.8 MB/s eta 0:00:00


In [ ]:
import torch
from diffusers import StableAudioPipeline

# Load model (half precision for GPU memory efficiency)
pipe = StableAudioPipeline.from_pretrained("stabilityai/stable-audio-open-1.0",torch_dtype=torch.float16,).to("cuda")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

scheduler_config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

...

transformer/diffusion_pytorch_model.safe(…):   0%|          | 0.00/4.23G [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/624M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


In [ ]:
# Install soundfile if not already present
!pip install -q soundfile

import torch
import soundfile as sf
from diffusers import StableAudioPipeline

# If you already loaded 'pipe' earlier, skip the from_pretrained/load lines and just reuse `pipe`.
# Otherwise (fresh session), uncomment the next two lines to load the pipeline:
# pipe = StableAudioPipeline.from_pretrained("stabilityai/stable-audio-open-1.0", torch_dtype=torch.float16)
# pipe = pipe.to("cuda")

prompt = "An upbeat music with bass boost"
negative_prompt = "low quality, distorted, garbled, clipped"

# Deterministic seed (optional)
generator = torch.Generator(device="cuda").manual_seed(42)

# Generate ~20 seconds (audio_end_in_s is in seconds; default audio_start_in_s=0.0)
out = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=100,        # quality vs speed
    guidance_scale=7.5,             # stronger adherence to prompt
    audio_end_in_s=20.0,            # desired end time in seconds
    num_waveforms_per_prompt=1,
    generator=generator,
    output_type="pt"                # returns torch.Tensor objects
)

# `out.audios` is a list of torch tensors; audio[0] shape is (channels, samples)
audio_tensor = out.audios[0]                    # torch.Tensor, e.g., (2, N)
audio_np = audio_tensor.T.float().cpu().numpy() # shape -> (samples, channels)

# Save using soundfile at the pipeline's sampling rate (usually 44100 Hz)
sf.write("sample1.wav", audio_np, pipe.vae.sampling_rate)

print("Saved sample1.wav — duration approx:", audio_np.shape[0] / pipe.vae.sampling_rate, "s")


  0%|          | 0/100 [00:00<?, ?it/s]

Saved sample1.wav — duration approx: 20.0 s


In [ ]:
from IPython.display import Audio
Audio("sample.wav")

In [ ]:
from IPython.display import Audio
Audio("sample1.wav")